# 07 - Model comparison and error analysis

This notebook reads the saved outputs of `scripts/run_pipeline.py` and does the
cross-model comparison. Nothing is refitted here, so the numbers are exactly those in
`outputs/metrics/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, pipeline

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

forecasts = pd.read_csv(config.FORECAST_DIR / "all_forecasts_extended.csv",
                        index_col=0, parse_dates=True)
results = pd.read_csv(config.METRICS_DIR / "model_comparison_annotated.csv")

print(open(config.METRICS_DIR / "run_info.json").read())

## The headline table

In [ ]:
VALID = [m for m in results["model"] if m != "feature_model_one_step"]

table = results.loc[results["model"].isin(VALID)].copy()
table[["model", "MAE", "RMSE", "MASE", "Bias"]].round(3)

`feature_model_one_step` is excluded from every ranking in this notebook. It scores well
(MASE 0.602) but only because it uses the previous hour's observation, which is unavailable
at a 24-hour horizon. Notebook 05 explains this in full.

## Ranking depends on the metric

In [ ]:
ranking = pd.DataFrame({
    "by MASE": table.sort_values("MASE")["model"].values,
    "by RMSE": table.sort_values("RMSE")["model"].values,
})

ranking.head(8)

The two metrics disagree at the top. Chronos-2 is first on MAE and MASE but only sixth on
RMSE, where the hour-of-week mean profile wins outright.

This is not a technicality. It reflects a real difference in behaviour: Chronos tracks
typical hours very closely and undershoots peaks badly, while the mean profile is mediocre
everywhere but never catastrophically wrong. Squared error punishes the former; absolute
error punishes the latter.

## Are the differences statistically significant?

A ranking on 336 observations is not much use without an indication of whether the gaps
could be noise. The Diebold-Mariano test compares two forecasts' absolute error series,
accounting for autocorrelation.

In [ ]:
dm = pd.read_csv(config.METRICS_DIR / "diebold_mariano.csv")
dm.round(4)

Against the strongest benchmark, only **Chronos-2 is significantly better** (DM statistic
2.36, p = 0.018). The feature model's 3.5% edge gives p = 0.53 and the SARIMAX model's
gives p = 0.99, so neither is distinguishable from the benchmark. The daily seasonal naive
forecast is significantly *worse* (p = 0.019).

The honest summary is that of three increasingly sophisticated model classes, exactly one
produces an improvement that survives a significance test, and the improvement is 14%.

## Visual comparison

In [ ]:
fig = plotting.plot_metric_comparison(results.loc[results["model"].isin(VALID)], metric="MASE")
fig

In [ ]:
headline = ["seasonal_mean_profile", "sarimax_target_only", "feature_model", "foundation_model"]

fig = plotting.plot_forecast_zoom(test, forecasts, columns=headline, days=4)
fig

The zoomed view shows the common failure. Every model reproduces the shape of the daily
cycle and every model undershoots the peaks. On 16 May the actual series reaches nearly
270 Wh; the closest forecast is around 180.

## The universal problem: peaks

In [ ]:
peak_hours = test > test.quantile(0.9)

summary = []
for model in headline:
    error = forecasts[model] - test
    summary.append({
        "model": model,
        "MAE all hours": error.abs().mean(),
        "MAE top-decile hours": error.abs()[peak_hours].mean(),
        "bias top-decile hours": error[peak_hours].mean(),
    })

pd.DataFrame(summary).round(1)

In [ ]:
print(f"Actual maximum in the test period: {test.max():.0f} Wh")
print("\nMaximum forecast by each model:")
for model in headline:
    print(f"  {model:<25} {forecasts[model].max():>6.0f} Wh")

The actual series reaches 495 Wh. No model forecasts above 254 Wh. In the top decile of
hours, every model has a bias between -130 and -190 Wh.

This is not a defect that better tuning would fix. All of these models estimate a
conditional mean or median, and the conditional distribution of appliance use in a peak
hour is genuinely wide: the household sometimes cooks at 18:00 and sometimes does not.
The optimal point forecast under squared or absolute loss is somewhere in the middle, so
peaks are underestimated by construction.

The practical response is not a better point forecast but a probabilistic one, which is
where Chronos's well-calibrated intervals matter.

## Interval quality

In [ ]:
intervals = pd.read_csv(config.METRICS_DIR / "interval_coverage.csv")
intervals.round(3)

Chronos: 77.7% coverage at 92 Wh width. SARIMAX: 91.1% coverage at 181 Wh.

Chronos is marginally under-covering and SARIMAX substantially over-covering, and Chronos
achieves it with intervals half as wide. On the interval evidence the foundation model is
clearly the better probabilistic forecaster.

## When are the errors made?

In [ ]:
by_hour = pd.read_csv(config.METRICS_DIR / "mae_by_hour.csv", index_col=0)
by_hour.round(1)

Errors are tiny overnight (3 to 5 Wh between 01:00 and 05:00) and an order of magnitude
larger between 10:00 and 20:00. Roughly speaking, the forecasting problem is trivial for
two thirds of the day and hard for the other third.

## A caution about error by horizon

`outputs/metrics/mae_by_horizon.csv` looks like it shows error growing with lead time. It
does not, and the reason is worth spelling out.

In [ ]:
by_horizon = pd.read_csv(config.METRICS_DIR / "mae_by_horizon.csv", index_col=0)

origin_hour = train.index[-1].hour
rotated = by_hour.reindex([(origin_hour + h) % 24 for h in range(1, 25)])

print(f"All origins fall at {origin_hour}:00, so step h always lands on hour ({origin_hour} + h) mod 24.")
print("Is the horizon table just the hour table, rotated?",
      np.allclose(rotated.to_numpy(), by_horizon.to_numpy()))

They are the same numbers. Because the design issues one forecast per day at a fixed time,
horizon and hour-of-day are perfectly confounded, and the apparent "error grows with
horizon" pattern is entirely a time-of-day effect.

`scripts/horizon_analysis.py` breaks the confound by re-issuing the feature model from
every hour in the test period, so each horizon step averages over all 24 clock times. Run
it and load the result here.

In [ ]:
horizon_path = config.METRICS_DIR / "horizon_analysis.csv"

if horizon_path.exists():
    corrected = pd.read_csv(horizon_path, index_col=0)
    display(corrected.round(2))
    slope = np.polyfit(corrected.index, corrected["MAE"], 1)[0]
    print(f"\nTrend: {slope:+.3f} Wh per additional hour of lead time")
    print(f"MAE at h=1:  {corrected['MAE'].iloc[0]:.1f} Wh")
    print(f"MAE at h=24: {corrected['MAE'].iloc[-1]:.1f} Wh")
else:
    print("Run: python scripts/horizon_analysis.py")

Once the confound is removed, error grows only mildly across the day: from 35.5 Wh at one
hour ahead to a maximum of 38.9 Wh at twenty hours ahead, a trend of about +0.10 Wh per hour
of lead time.

Note also that accuracy recovers at the longest horizons, falling back to 37.5 Wh at h = 24.
At a 24-hour horizon the latest observation available at the origin is exactly the same hour
on the previous day, so `origin_lag_0` coincides with the daily seasonal lag and becomes
informative again. Predictability is best when the origin is either very recent or exactly
one seasonal period away.

That is a strong statement about this series: almost all of the predictable signal is the
seasonal profile, which is equally available at any lead time, and almost none of it is
short-run momentum that decays as the horizon extends.

## Sensitivity to the evaluation design

In [ ]:
single = pd.read_csv(config.METRICS_DIR / "model_comparison_single_block.csv")

pd.merge(
    results[["model", "MASE"]].rename(columns={"MASE": "rolling_origin_24h"}),
    single[["model", "MASE"]].rename(columns={"MASE": "single_block_336h"}),
    on="model",
).round(3)

The seasonal methods and all three model classes are essentially unchanged. Only `naive`
and `drift` differ, and they differ enormously.

The reassuring conclusion is that the substantive findings do not depend on the choice
between the two designs. The one thing the design does change is how bad the worst
benchmarks look, which is a reason to be sceptical of any study that reports large
improvements over a naive benchmark evaluated over a long single block.

## Summary of findings

1. The strongest benchmark is the hour-of-week mean profile at MASE 0.712, beating both
   seasonal naive variants.
2. SARIMAX improves on it by 4% (0.682), which is not statistically significant.
3. The feature-based model improves on it by 3.5% (0.687), also not significant.
4. Chronos-2 zero-shot improves on it by 14% (0.614), which **is** significant (p = 0.018),
   and produces far better calibrated intervals.
5. Every covariate set tried made forecasts worse.
6. All models substantially underestimate peaks, and no model exceeds 254 Wh against an
   observed maximum of 495 Wh.